# BCWD GH-ANFIS 5-Fold Training

Breast Cancer Wisconsin (Original) 데이터셋에 대해 `GH-ANFIS`만 5-fold 교차검증으로 학습하는 전용 노트북입니다.

- 입력은 `load_bcwd_data()`를 그대로 사용하므로, E404 전처리 기준에서 BCWD는 one-hot 확장 없이 `9`개 numeric feature로 학습됩니다.
- 하이퍼파라미터는 기본적으로 `hyper_parameter/best_GH-ANFIS_HP.json`의 BCWD 항목을 로드합니다.
- fold별 체크포인트는 `hyper_parameter/cv_weights/Breast_Cancer_Wisconsin__Original___<mode>/fold_XX/gh_anfis.pt`에 저장됩니다.
- 요약 CSV와 best-fold gate 테이블은 `output/bcwd_gh_anfis/` 아래에 저장됩니다.


In [ ]:
from pathlib import Path
import copy
import os
import sys

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler


PROJECT_ROOT = Path.cwd().resolve()


def _looks_like_project_root(path):
    markers = ["model.py", "data.py", "learning.py", "utils.py"]
    return all((path / marker).exists() for marker in markers)


if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not _looks_like_project_root(PROJECT_ROOT):
    candidate_roots = [
        PROJECT_ROOT / "03_Research" / "GH-ANFIS_E404",
        PROJECT_ROOT / "03_Research" / "GH-ANFIS_exp",
    ]
    for cand in candidate_roots:
        if _looks_like_project_root(cand):
            PROJECT_ROOT = cand
            break
    else:
        for parent in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
            if _looks_like_project_root(parent):
                PROJECT_ROOT = parent
                break

os.chdir(PROJECT_ROOT)

os.environ.setdefault("OPENML_DATA_HOME", str(PROJECT_ROOT / "data" / "openml_cache"))
os.environ.setdefault("XDG_CACHE_HOME", str(PROJECT_ROOT / ".cache"))
(PROJECT_ROOT / "data" / "openml_cache").mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / ".cache").mkdir(parents=True, exist_ok=True)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from data import load_bcwd_data, coerce_numeric_frame, drop_nan_targets
from learning import fit_gh_anfis
from gh_eval_utils import evaluate_torch_classification
from utils import set_deterministic, build_loader, load_gh_params

SEED = 42
set_deterministic(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CV_WEIGHT_ROOT = Path("./hyper_parameter/cv_weights")
MODEL_NAME = "GH-ANFIS"


def _safe_name(name):
    return "".join(ch if ch.isalnum() or ch in ("-", "_") else "_" for ch in str(name))


def _scaler_stats(scaler):
    if scaler is None:
        return None, None
    mean = getattr(scaler, "mean_", None)
    scale = getattr(scaler, "scale_", None)
    return (mean.tolist() if mean is not None else None, scale.tolist() if scale is not None else None)


def save_cv_torch_artifact(
    dataset_name,
    model_name,
    fold_idx,
    model,
    params,
    task_kind,
    n_outputs,
    n_features,
    scaler=None,
    feature_names=None,
    root_dir=CV_WEIGHT_ROOT,
    extra_meta=None,
):
    dataset_key = _safe_name(dataset_name)
    fold_dir = Path(root_dir) / dataset_key / f"fold_{int(fold_idx):02d}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    scaler_mean, scaler_scale = _scaler_stats(scaler)
    out_path = fold_dir / "gh_anfis.pt"

    state_dict_cpu = {
        k: (v.detach().cpu() if torch.is_tensor(v) else v)
        for k, v in model.state_dict().items()
    }

    payload = {
        "framework": "torch",
        "dataset": str(dataset_name),
        "dataset_key": dataset_key,
        "model_name": model_name,
        "fold": int(fold_idx),
        "task_kind": str(task_kind),
        "n_outputs": int(n_outputs),
        "n_features": int(n_features),
        "feature_names": list(feature_names) if feature_names is not None else None,
        "params": copy.deepcopy(params),
        "scaler_mean": scaler_mean,
        "scaler_scale": scaler_scale,
        "state_dict": state_dict_cpu,
        "extra_meta": dict(extra_meta or {}),
    }
    torch.save(payload, out_path)
    return str(out_path)


print(f"Project root: {PROJECT_ROOT}")
PROJECT_ROOT, DEVICE


In [ ]:
DATASET_NAME = 'Breast_Cancer_Wisconsin_(Original)'
TASK_KIND = 'binary'
N_OUTPUTS = 1
MODE_LABEL = 'gh_only_5fold'
N_FOLDS = 5
MAX_FOLDS = None  # smoke test가 필요하면 1처럼 줄여서 실행
BATCH_SIZE = 1024
USE_JSON_BEST_PARAMS = True
GH_PARAM_OVERRIDES = {}

OUTPUT_DIR = PROJECT_ROOT / 'output' / 'bcwd_gh_anfis'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

X_df, y, feature_names = load_bcwd_data()
X_df = coerce_numeric_frame(X_df)
X_df, y = drop_nan_targets(X_df, y)
X_df = X_df.copy()
y = np.asarray(y, dtype=np.int64)
feature_names = list(X_df.columns)

print('X shape:', X_df.shape)
print('y shape:', y.shape)
print('n_features:', len(feature_names))
print('class counts:')
print(pd.Series(y).value_counts().sort_index())
print('feature names:', feature_names)


In [ ]:
if USE_JSON_BEST_PARAMS:
    GH_PARAMS = load_gh_params(DATASET_NAME, DATASET_NAME)
else:
    GH_PARAMS = {
        'lr_base': 0.1,
        'lr_residual': 0.005,
        'residual_rules': 11,
        'base_rules': 6,
        'mf_per_feature': 2,
        'epochs_stage1': 80,
        'epochs_stage2': 40,
        'lambda_resid_s2': 0.0,
        'lambda_base_s1': 0.01,
        'weight_decay': 1e-05,
        'base_hard_epochs': 30,
        'residual_hard_epochs': 20,
        'base_mask_threshold': None,
        'residual_mask_threshold': None,
        'lambda_base_hard': 0.0,
        'lambda_resid_hard': 0.0,
        'rule_init_mode': 'balanced',
        'rule_seed': 0,
        'firing_mode': 'htsk',
        'residual_gate_mode': 'complement',
        'use_input_norm': False,
        'enable_residual_branch': True,
        'random_role_assignment': False,
    }

GH_PARAMS = copy.deepcopy(GH_PARAMS)
GH_PARAMS.update(GH_PARAM_OVERRIDES)
GH_PARAMS['rule_seed'] = int(GH_PARAMS.get('rule_seed', SEED))
GH_PARAMS.setdefault('rule_init_mode', 'balanced')
GH_PARAMS.setdefault('firing_mode', 'htsk')
GH_PARAMS.setdefault('residual_gate_mode', 'complement')
GH_PARAMS.setdefault('use_input_norm', False)
GH_PARAMS.setdefault('enable_residual_branch', True)
GH_PARAMS.setdefault('random_role_assignment', False)

pd.Series(GH_PARAMS).sort_index()


In [ ]:
eval_settings = [
    ('base_hard', 'base', 'base_only', False),
    ('base_soft', 'base', 'base_only', True),
    ('residual_hard', 'residual_complement', 'residual_only', False),
    ('full_hard', 'residual_complement', 'full', False),
    ('full_soft', 'residual_complement', 'full', True),
]


def run_bcwd_gh_cv(dataset_name, params, max_folds=None, mode_label='gh_only_5fold'):
    dataset_artifact_name = f"{dataset_name}__{mode_label}"
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

    fold_rows = []
    saved_artifact_paths = []
    best_fold_idx = None
    best_fold_full_hard_f1 = -np.inf
    best_fold_results_df = None
    best_fold_gate_df = None

    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_df, y), 1):
        if max_folds is not None and int(fold_idx) > int(max_folds):
            break

        set_deterministic(SEED)
        print('=' * 60)
        print(f"[{dataset_name}/{mode_label}] Fold {fold_idx}/{N_FOLDS}")
        print('=' * 60)

        X_train_df = X_df.iloc[train_idx].copy()
        X_val_df = X_df.iloc[val_idx].copy()
        y_train = y[train_idx]
        y_val = y[val_idx]

        scaler = StandardScaler().fit(X_train_df)
        X_train = scaler.transform(X_train_df).astype(np.float32)
        X_val = scaler.transform(X_val_df).astype(np.float32)

        train_loader = build_loader(X_train, y_train, BATCH_SIZE, DEVICE, TASK_KIND)
        val_loader = build_loader(X_val, y_val, BATCH_SIZE, DEVICE, TASK_KIND)

        fold_params = copy.deepcopy(params)
        model, criterion = fit_gh_anfis(
            fold_params,
            train_loader,
            X_train.shape[1],
            N_OUTPUTS,
            TASK_KIND,
            feature_names,
            DEVICE,
            verbose=True,
        )

        fold_results = []
        for label, phase, mode, use_soft_eval in eval_settings:
            model.set_phase(phase)
            model.set_mode(mode)
            loss, acc, f1 = evaluate_torch_classification(
                model,
                val_loader,
                criterion,
                DEVICE,
                TASK_KIND,
                use_soft_eval=use_soft_eval,
            )
            row = {
                'fold': fold_idx,
                'view': label,
                'phase': phase,
                'mode': mode,
                'use_soft_eval': use_soft_eval,
                'loss': loss,
                'acc': acc,
                'f1': f1,
                'train_size': int(X_train.shape[0]),
                'val_size': int(X_val.shape[0]),
            }
            fold_results.append(row)
            fold_rows.append(row)

        base_probs = torch.sigmoid(model.base_mask_logits).detach().cpu().numpy()
        residual_probs = torch.sigmoid(model.residual_mask_logits).detach().cpu().numpy()
        base_hard = model.base_mask_hard.detach().cpu().numpy()
        residual_hard = model.residual_mask_hard.detach().cpu().numpy()
        residual_effective_hard = residual_hard * (1.0 - base_hard)

        gate_df = pd.DataFrame(
            {
                'fold': fold_idx,
                'feature': feature_names,
                'base_prob': base_probs,
                'base_hard': base_hard,
                'residual_prob': residual_probs,
                'residual_hard': residual_hard,
                'residual_effective_hard': residual_effective_hard,
            }
        )
        gate_df = gate_df.sort_values(
            ['base_hard', 'base_prob', 'residual_effective_hard', 'residual_prob'],
            ascending=[False, False, False, False],
        ).reset_index(drop=True)

        full_hard_row = next(row for row in fold_results if row['view'] == 'full_hard')
        if full_hard_row['f1'] > best_fold_full_hard_f1:
            best_fold_full_hard_f1 = float(full_hard_row['f1'])
            best_fold_idx = int(fold_idx)
            best_fold_results_df = pd.DataFrame(fold_results)
            best_fold_gate_df = gate_df.copy()

        artifact_path = save_cv_torch_artifact(
            dataset_name=dataset_artifact_name,
            model_name=MODEL_NAME,
            fold_idx=fold_idx,
            model=model,
            params=fold_params,
            task_kind=TASK_KIND,
            n_outputs=N_OUTPUTS,
            n_features=X_train.shape[1],
            scaler=scaler,
            feature_names=feature_names,
            extra_meta={
                'seed': SEED,
                'batch_size': BATCH_SIZE,
                'train_size': int(X_train.shape[0]),
                'val_size': int(X_val.shape[0]),
                'positive_ratio_train': float(np.mean(y_train)),
                'positive_ratio_val': float(np.mean(y_val)),
                'fold_results': fold_results,
                'gate_rows': gate_df.to_dict(orient='records'),
            },
        )
        saved_artifact_paths.append(artifact_path)
        print(f"Saved fold artifacts: {Path(artifact_path).parent}")

    fold_results_df = pd.DataFrame(fold_rows)
    summary_rows = []
    for view in [cfg[0] for cfg in eval_settings]:
        view_df = fold_results_df[fold_results_df['view'] == view]
        summary_rows.append({
            'view': view,
            'loss_mean': float(view_df['loss'].mean()),
            'loss_std': float(view_df['loss'].std(ddof=0)),
            'acc_mean': float(view_df['acc'].mean()),
            'acc_std': float(view_df['acc'].std(ddof=0)),
            'f1_mean': float(view_df['f1'].mean()),
            'f1_std': float(view_df['f1'].std(ddof=0)),
        })
    summary_df = pd.DataFrame(summary_rows)

    return {
        'dataset_artifact_name': dataset_artifact_name,
        'saved_artifact_paths': saved_artifact_paths,
        'fold_results_df': fold_results_df,
        'summary_df': summary_df,
        'best_fold_idx': best_fold_idx,
        'best_fold_results_df': best_fold_results_df,
        'best_fold_gate_df': best_fold_gate_df,
    }


In [ ]:
run_output = run_bcwd_gh_cv(
    dataset_name=DATASET_NAME,
    params=copy.deepcopy(GH_PARAMS),
    max_folds=MAX_FOLDS,
    mode_label=MODE_LABEL,
)

DATASET_ARTIFACT_NAME = run_output['dataset_artifact_name']
saved_artifact_paths = run_output['saved_artifact_paths']
fold_results_df = run_output['fold_results_df']
summary_df = run_output['summary_df']
best_fold_idx = run_output['best_fold_idx']
best_fold_results_df = run_output['best_fold_results_df']
best_fold_gate_df = run_output['best_fold_gate_df']

summary_df


In [ ]:
print(f'Best fold by full_hard f1: {best_fold_idx}')
best_fold_results_df


In [ ]:
best_fold_gate_df


In [ ]:
summary_path = OUTPUT_DIR / f'bcwd_{MODE_LABEL}_summary.csv'
fold_results_path = OUTPUT_DIR / f'bcwd_{MODE_LABEL}_fold_results.csv'
best_fold_results_path = OUTPUT_DIR / f'bcwd_{MODE_LABEL}_best_fold_results.csv'
best_fold_gate_path = OUTPUT_DIR / f'bcwd_{MODE_LABEL}_best_fold_gate.csv'
artifact_paths_path = OUTPUT_DIR / f'bcwd_{MODE_LABEL}_artifact_paths.csv'

summary_df.to_csv(summary_path, index=False)
fold_results_df.to_csv(fold_results_path, index=False)
best_fold_results_df.to_csv(best_fold_results_path, index=False)
best_fold_gate_df.to_csv(best_fold_gate_path, index=False)
pd.DataFrame({'artifact_path': saved_artifact_paths}).to_csv(artifact_paths_path, index=False)

print('Saved summary:', summary_path)
print('Saved fold results:', fold_results_path)
print('Saved best-fold results:', best_fold_results_path)
print('Saved best-fold gate:', best_fold_gate_path)
print('Saved artifact paths:', artifact_paths_path)
